# PlacementIQ — EDA and Data Cleaning

**Goal:** explore `student_placement_train.csv`, document data-quality issues, and create a conservative cleaned copy for the later ML pipeline.

## Important methodology
- Keep the original CSV unchanged.
- Do not impute missing values, scale features, or encode categories across the full dataset. Those steps belong inside a scikit-learn pipeline fitted on the training split.
- Do not delete outliers automatically. Investigate them first and document any justified correction.
- `student_id` is an identifier, not a predictive feature.
- `placed` is the target (expected values: `0` and `1`).
- The dataset may be imbalanced; report class counts and use appropriate metrics later.

## 1. Load libraries and dataset

In Google Colab, upload `student_placement_train.csv` when prompted. In local Jupyter, place the CSV in the same folder as this notebook or update `CSV_PATH`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

CSV_PATH = Path("student_placement_train.csv")

if not CSV_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file uploaded. Upload student_placement_train.csv.")
        CSV_PATH = Path(csv_files[0])
    except ImportError:
        raise FileNotFoundError(
            "CSV not found. Put student_placement_train.csv next to this notebook "
            "or update CSV_PATH to its location."
        )

df_raw = pd.read_csv(CSV_PATH)
df = df_raw.copy()

print(f"Loaded file: {CSV_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 2. Schema and basic quality checks

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
display(df.info())

print("\nDescriptive statistics — numeric columns:")
display(df.describe().T)

print("\nDescriptive statistics — categorical columns:")
display(df.describe(include=["object", "category", "bool"]).T)

## 3. Check expected columns and target

In [ ]:
expected_columns = [
    "student_id", "age", "gender", "cgpa", "attendance_percentage", "backlogs",
    "coding_score", "aptitude_score", "communication_score", "technical_score",
    "projects_count", "major_projects", "internships_count", "internship_months",
    "certifications_count", "hackathons_participated", "hackathons_won",
    "coding_platform_score", "github_projects", "linkedin_score", "resume_score",
    "soft_skills_score", "leadership_score", "extracurricular_score", "training_hours",
    "mock_interview_score", "preferred_domain", "placed"
]

missing_expected = sorted(set(expected_columns) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(expected_columns))

print("Expected columns missing from file:", missing_expected or "None")
print("Unexpected columns:", unexpected_columns or "None")

TARGET = "placed"
ID_COLUMNS = ["student_id"]

if TARGET not in df.columns:
    raise ValueError("The expected target column 'placed' is not present. Check the dataset schema.")

print("\nTarget value counts (including missing values):")
display(df[TARGET].value_counts(dropna=False).rename_axis("placed").to_frame("count"))

## 4. Missing values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_percent", ascending=False)

display(missing_summary[missing_summary["missing_count"] > 0])

if missing_summary["missing_count"].sum() == 0:
    print("No missing values detected.")

missing_to_plot = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_percent")
if not missing_to_plot.empty:
    plt.figure(figsize=(10, max(4, 0.35 * len(missing_to_plot))))
    plt.barh(missing_to_plot.index, missing_to_plot["missing_percent"])
    plt.xlabel("Missing values (%)")
    plt.title("Missing-value percentage by column")
    plt.tight_layout()
    plt.show()

## 5. Duplicates and identifier checks

In [ ]:
exact_duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows:", exact_duplicate_count)

for col in ID_COLUMNS:
    if col in df.columns:
        print(f"\n{col}:")
        print("Missing IDs:", int(df[col].isna().sum()))
        print("Duplicate non-null IDs:", int(df[col].dropna().duplicated().sum()))
        print("Unique non-null IDs:", int(df[col].nunique(dropna=True)))

print("\nRows with duplicate student_id (if the ID column exists):")
if "student_id" in df.columns:
    display(df[df["student_id"].duplicated(keep=False) & df["student_id"].notna()]
            .sort_values("student_id").head(20))

## 6. Target class balance

In [ ]:
target_counts = df[TARGET].value_counts(dropna=False).sort_index()
display(target_counts.rename("count").to_frame())

valid_target = df[TARGET].dropna()
print("\nTarget labels found:", sorted(valid_target.unique().tolist(), key=str))

if len(valid_target) > 0:
    target_percent = (valid_target.value_counts(normalize=True) * 100).sort_index()
    display(target_percent.rename("percent").round(2).to_frame())

    plt.figure(figsize=(6, 4))
    counts_for_plot = valid_target.value_counts().sort_index()
    plt.bar(counts_for_plot.index.astype(str), counts_for_plot.values)
    plt.xlabel("placed")
    plt.ylabel("Number of records")
    plt.title("Placement target distribution")
    plt.tight_layout()
    plt.show()

print("Important: if the classes are imbalanced, do not rely on accuracy alone.")

## 7. Numeric feature distributions

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_features = [c for c in numeric_cols if c != TARGET and c not in ID_COLUMNS]

print(f"Numeric columns: {len(numeric_cols)}")
display(df[numeric_features].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T)

if numeric_features:
    df[numeric_features].hist(figsize=(18, 14), bins=30, edgecolor="black")
    plt.suptitle("Numeric feature distributions", y=1.02)
    plt.tight_layout()
    plt.show()

## 8. Outlier screening (flag only; do not automatically delete)

In [ ]:
outlier_rows = []
for col in numeric_features:
    series = df[col].dropna()
    if series.nunique() < 5:
        continue
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        lower, upper = series.quantile(0.01), series.quantile(0.99)
    else:
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((series < lower) | (series > upper)).sum())
    outlier_rows.append({
        "column": col, "lower_screening_bound": lower,
        "upper_screening_bound": upper, "flagged_count": count,
        "flagged_percent_of_nonmissing": round(100 * count / len(series), 2)
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values("flagged_count", ascending=False)
display(outlier_summary)

print(
    "These are statistical flags, not proof of data errors. Check domain ranges and source definitions "
    "before correcting or removing any observation."
)

## 9. Categorical feature distributions

In [ ]:
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
categorical_features = [c for c in categorical_cols if c not in ID_COLUMNS]

for col in categorical_features:
    print(f"\n{col}: {df[col].nunique(dropna=False)} distinct values")
    display(df[col].value_counts(dropna=False).head(20).rename("count").to_frame())

## 10. Correlations and relationship with placement

In [ ]:
if len(numeric_features) >= 2:
    corr_cols = numeric_features + ([TARGET] if TARGET in df.columns and pd.api.types.is_numeric_dtype(df[TARGET]) else [])
    corr = df[corr_cols].corr(numeric_only=True)
    plt.figure(figsize=(14, 10))
    plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Numeric feature correlation matrix")
    plt.tight_layout()
    plt.show()

if TARGET in df.columns:
    for col in ["cgpa", "aptitude_score", "technical_score", "communication_score",
                "internships_count", "projects_count", "mock_interview_score"]:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            grouped = df.groupby(TARGET)[col].median()
            print(f"Median {col} by placed class:")
            display(grouped.rename("median").to_frame())

    if "preferred_domain" in df.columns:
        domain_rate = df.groupby("preferred_domain", dropna=False)[TARGET].agg(["count", "mean"])
        domain_rate = domain_rate.rename(columns={"mean": "placement_rate"}).sort_values("placement_rate", ascending=False)
        display(domain_rate)

## 11. Domain checks for suspicious values

In [ ]:
# These are common-sense screening ranges, not automatic corrections.
# Confirm the source's definitions before deciding whether any flagged value is invalid.
candidate_ranges = {
    "age": (14, 100),
    "cgpa": (0, 10),
    "attendance_percentage": (0, 100),
    "coding_score": (0, 100),
    "aptitude_score": (0, 100),
    "communication_score": (0, 100),
    "technical_score": (0, 100),
    "linkedin_score": (0, 100),
    "resume_score": (0, 100),
    "soft_skills_score": (0, 100),
    "leadership_score": (0, 100),
    "extracurricular_score": (0, 100),
    "mock_interview_score": (0, 100),
    "backlogs": (0, None),
    "projects_count": (0, None),
    "major_projects": (0, None),
    "internships_count": (0, None),
    "internship_months": (0, None),
    "certifications_count": (0, None),
    "hackathons_participated": (0, None),
    "hackathons_won": (0, None),
    "github_projects": (0, None),
    "training_hours": (0, None),
}

range_issues = []
for col, (lower, upper) in candidate_ranges.items():
    if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
        continue
    mask = pd.Series(False, index=df.index)
    if lower is not None:
        mask |= df[col].notna() & (df[col] < lower)
    if upper is not None:
        mask |= df[col].notna() & (df[col] > upper)
    range_issues.append({
        "column": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "flagged_count": int(mask.sum())
    })

range_summary = pd.DataFrame(range_issues)
display(range_summary)

print(
    "Review flagged rows with the dataset documentation. Do not assume these ranges are authoritative "
    "if the source uses different score scales."
)

## 12. Cleaning decisions and create a conservative cleaned copy

In [ ]:
# Preserve the raw data. Remove exact duplicate rows only in the derived copy.
# Do not globally impute, scale or encode: those operations must be fitted on training data.
df_clean = df.copy()
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
exact_duplicates_removed = rows_before - len(df_clean)

# Normalize whitespace for object/string columns without changing the original raw dataframe.
for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[col] = df_clean[col].map(lambda x: x.strip() if isinstance(x, str) else x)

# Check target values, but do not silently drop invalid target rows.
valid_target_values = {0, 1}
observed_nonmissing_targets = set(df_clean[TARGET].dropna().unique().tolist())
invalid_target_values = observed_nonmissing_targets - valid_target_values

print("Rows before:", rows_before)
print("Exact duplicate rows removed:", exact_duplicates_removed)
print("Rows in conservative cleaned copy:", len(df_clean))
print("Target missing values:", int(df_clean[TARGET].isna().sum()))
print("Unexpected target values:", invalid_target_values)

if invalid_target_values:
    print("ACTION REQUIRED: investigate unexpected target values before model training.")
if df_clean[TARGET].isna().any():
    print("ACTION REQUIRED: investigate rows with missing target; do not impute the target.")

print("\nMissing values remain in the cleaned copy by design:")
display(df_clean.isna().sum().sort_values(ascending=False).head(15).rename("missing_count").to_frame())

## 13. Export cleaned data and audit summary

In [ ]:
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = OUTPUT_DIR / "student_placement_cleaned.csv"
summary_path = OUTPUT_DIR / "eda_audit_summary.json"
missing_path = OUTPUT_DIR / "missing_value_summary.csv"
outlier_path = OUTPUT_DIR / "outlier_screening_summary.csv"

df_clean.to_csv(cleaned_path, index=False)
missing_summary.to_csv(missing_path)
outlier_summary.to_csv(outlier_path, index=False)

audit_summary = {
    "source_file": str(CSV_PATH),
    "original_rows": int(len(df_raw)),
    "original_columns": int(df_raw.shape[1]),
    "cleaned_rows": int(len(df_clean)),
    "cleaned_columns": int(df_clean.shape[1]),
    "exact_duplicate_rows_removed": int(exact_duplicates_removed),
    "target_column": TARGET,
    "target_missing_count": int(df_clean[TARGET].isna().sum()),
    "target_values": [str(x) for x in df_clean[TARGET].dropna().unique().tolist()],
    "student_id_excluded_from_model_features": True,
    "missing_values_imputed": False,
    "scaling_or_encoding_applied": False,
    "notes": [
        "Original CSV preserved.",
        "Only exact duplicate rows were removed from the derived copy.",
        "Missing values intentionally retained for pipeline-based training-only imputation.",
        "Outlier flags require domain review; no outliers automatically removed.",
        "Exclude student_id and placed from predictive features.",
        "Review gender inclusion as a fairness decision before modeling."
    ]
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(audit_summary, f, indent=2)

print("Saved files:")
print("-", cleaned_path)
print("-", summary_path)
print("-", missing_path)
print("-", outlier_path)
display(pd.DataFrame([audit_summary]))

## 14. Final checklist before ML training

- [ ] Confirm the dataset source and license.
- [ ] Confirm `placed` contains only 0 and 1, with no missing target values.
- [ ] Review duplicate IDs and exact duplicates.
- [ ] Review the missing-value summary.
- [ ] Investigate range flags and outliers; do not remove them blindly.
- [ ] Exclude `student_id` and `placed` from X.
- [ ] Document the decision to exclude or include sensitive/proxy variables.
- [ ] Use stratified splitting.
- [ ] Fit imputers, encoders and scalers only inside training pipelines.
- [ ] Evaluate all four models using precision, recall, F1, ROC-AUC and balanced accuracy, not accuracy alone.

**Next step:** use `student_placement_cleaned.csv` as the reviewed input if the conservative cleaning decisions are appropriate. The ML training code must still split the data first and perform imputation, encoding and scaling inside each training pipeline.